# Ejercicio 05 · Motor recomendador reproducible

Consolida en un único producto ejecutable las evidencias de LAB08 (corpus, auditoría, baseline de
popularidad y recomendación por contenido) y LAB09 (factorización, corte temporal, sistema híbrido,
usuarios fríos y frontera de Pareto).

**Pregunta.** Sobre MovieLens *Latest Small*, ¿qué relación hay entre la señal colaborativa
(factorización), la señal de contenido (géneros) y la popularidad, y qué configuración ofrece la
mejor relación entre acierto en el ranking y cobertura del catálogo?

**Datos y licencia.** MovieLens es de **GroupLens Research, University of Minnesota**; se usa con
fines académicos citando la fuente (Harper y Konstan, 2015, DOI 10.1145/2827872). **No se
redistribuye**: el dataset se descarga por script y no se versiona en el repositorio.

**Protocolo.** Corte temporal *leave-one-out* (la última valoración de cada usuario queda como
prueba), semilla 42, métricas de ranking (HitRate@10) y de catálogo (cobertura).

In [1]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u03.config import MOVIELENS_DIR, RANDOM_STATE
from inf8239_u03.data import download_zip, load_movielens, validate_movielens
from inf8239_u03.hybrid import build_genre_features, evaluate_hybrid, hybrid_top_n
from inf8239_u03.metrics import catalog_coverage, hit_rate_at_k
from inf8239_u03.recommenders import (
    ContentRecommender,
    MatrixFactorization,
    temporal_leave_one_out,
    weighted_popularity,
)

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
print("Proyecto:", ROOT, "| semilla:", RANDOM_STATE)

Proyecto: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_3/INF8239_U03_Recomendadores | semilla: 42


## 1. Descarga reproducible y auditoría

El script descarga el ZIP oficial de GroupLens y valida que no haya rutas inseguras; después se
registra el SHA-256 para dejar trazabilidad del corpus exacto con el que se entrenó.

In [2]:
from inf8239_u03.config import MOVIELENS_URL

if not MOVIELENS_DIR.exists():
    path, digest = download_zip(MOVIELENS_URL, MOVIELENS_DIR)
    print("Dataset descargado:", path)
    print("SHA-256:", digest)
else:
    print("Dataset ya presente en:", MOVIELENS_DIR)

ratings, movies = load_movielens(MOVIELENS_DIR)
validate_movielens(ratings, movies)
print("Valoraciones:", ratings.shape, "| Películas:", movies.shape)
print("Usuarios:", ratings['userId'].nunique(), "| Ítems valorados:", ratings['movieId'].nunique())
print("Densidad: %.4f%%" % (len(ratings) / (ratings['userId'].nunique() * movies['movieId'].nunique()) * 100))
print("Rango temporal:", ratings['timestamp'].min(), "->", ratings['timestamp'].max())
print("Media de valoración: %.4f | nulos: %d" % (ratings['rating'].mean(), int(ratings.isna().sum().sum())))

Dataset ya presente en: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_3/INF8239_U03_Recomendadores/data/raw/ml-latest-small
Valoraciones: (100836, 4) | Películas: (9742, 3)
Usuarios: 610 | Ítems valorados: 9724
Densidad: 1.6968%
Rango temporal: 828124615 -> 1537799250
Media de valoración: 3.5016 | nulos: 0


## 2. Baseline de popularidad (LAB08)

La media de la película se combina con la media global, ponderada por el número de valoraciones: una
película con una sola valoración alta no debe dominar la lista.

In [3]:
popular = weighted_popularity(ratings, movies).head(10)
popular[["title", "count", "mean", "weighted_score"]].to_csv(REPORTS / "popular_top10.csv", index=False)

figura, eje = plt.subplots(figsize=(9, 5))
rotulo = [t if len(t) < 34 else t[:32] + "..." for t in popular["title"]]
eje.barh(rotulo[::-1], popular["weighted_score"][::-1], color="#0f7c7b")
eje.set_xlabel("Puntuación de popularidad (media global y media de la película ponderadas)")
eje.set_title("Top-10 de popularidad · MovieLens Latest Small")
eje.grid(axis="x", alpha=0.3)
figura.tight_layout()
figura.savefig(REPORTS / "popular_top10.png", dpi=170, bbox_inches="tight")
plt.close(figura)

popular[["title", "count", "mean", "weighted_score"]]

,title,count,mean,weighted_score
movieId,,,,
318,"Shawshank Redemption, The (1994)",317,4.429022,4.395194
858,"Godfather, The (1972)",192,4.289062,4.242739
2959,Fight Club (1999),218,4.272936,4.232690
260,Star Wars: Episode IV - A New Hope (1977),251,4.231076,4.197790
50,"Usual Suspects, The (1995)",204,4.237745,4.196846
1221,"Godfather: Part II, The (1974)",129,4.259690,4.195168
527,Schindler's List (1993),220,4.225000,4.187581
1213,Goodfellas (1990),126,4.250000,4.184918
750,Dr. Strangelove or: How I Learned to Stop Worr...,97,4.268041,4.183658


## 3. Recomendación por contenido (LAB08)

TF-IDF sobre los géneros y similitud del coseno. La similitud mide **coincidencia de géneros
registrados**, no afinidad de gusto: es la lectura obligatoria de este resultado.

In [4]:
recomendador = ContentRecommender().fit(movies)
consultas = {}
for titulo in ["Toy Story (1995)", "Matrix, The (1999)", "Godfather, The (1972)"]:
    tabla = recomendador.recommend(titulo, 10)
    consultas[titulo] = tabla
    tabla.to_csv(REPORTS / f"content_{titulo.split(',')[0].split('(')[0].strip().lower().replace(' ', '_')}.csv", index=False)
    print(f"\nSimilares a {titulo}:")
    print(tabla[["title", "genres", "content_score"]].to_string(index=False))


Similares a Toy Story (1995):
                                                  title                                      genres  content_score
                                           Turbo (2013) Adventure|Animation|Children|Comedy|Fantasy            1.0
                       Emperor's New Groove, The (2000) Adventure|Animation|Children|Comedy|Fantasy            1.0
         Adventures of Rocky and Bullwinkle, The (2000) Adventure|Animation|Children|Comedy|Fantasy            1.0
                                     Toy Story 2 (1999) Adventure|Animation|Children|Comedy|Fantasy            1.0
Asterix and the Vikings (Astérix et les Vikings) (2006) Adventure|Animation|Children|Comedy|Fantasy            1.0
                                            Antz (1998) Adventure|Animation|Children|Comedy|Fantasy            1.0
                                       Wild, The (2006) Adventure|Animation|Children|Comedy|Fantasy            1.0
                                 Shrek the Third 

## 4. Corte temporal y factorización de matriz (LAB09)

La partición **no** es aleatoria: se reserva la última valoración de cada usuario. Una separación
aleatoria dejaría que el modelo aprendiera de interacciones posteriores.

In [5]:
train, test = temporal_leave_one_out(ratings)
print("Entrenamiento:", train.shape, "| Prueba:", test.shape)
print("Usuarios evaluables en la prueba:", test['userId'].nunique())
print("Ejemplo de corte temporal (usuario 414):")
print(train[train['userId'].eq(414)].tail(2)[["userId", "movieId", "rating", "timestamp"]].to_string(index=False))
print("Su última valoración (queda en la prueba):")
print(test[test['userId'].eq(414)][["userId", "movieId", "rating", "timestamp"]].to_string(index=False))

Entrenamiento: (100226, 4) | Prueba: (610, 4)
Usuarios evaluables en la prueba: 610
Ejemplo de corte temporal (usuario 414):
 userId  movieId  rating  timestamp
    414   175661     3.0 1527977909
    414   187595     3.5 1527977920
Su última valoración (queda en la prueba):
 userId  movieId  rating  timestamp
    414   180985     3.5 1527978072


In [6]:
inicio = time.perf_counter()
modelo = MatrixFactorization(factors=20, seed=RANDOM_STATE).fit(train, epochs=12)
print("Tiempo de ajuste (segundos): %.2f" % (time.perf_counter() - inicio))

genre_matrix, movie_row = build_genre_features(movies)
for alfa in (0.75, 0.25):
    evaluacion = evaluate_hybrid(modelo, train, test, genre_matrix, movie_row, alfa)
    print(f"\nalpha={alfa} · RMSE={evaluacion['rmse']:.4f} · HitRate@10={evaluacion['hit_rate_at_10']:.5f} "
          f"· cobertura={evaluacion['catalog_coverage']:.4f} · usuarios evaluados={evaluacion['evaluable_users']}")

Tiempo de ajuste (segundos): 7.50



alpha=0.75 · RMSE=1.0264 · HitRate@10=0.03748 · cobertura=0.0775 · usuarios evaluados=587



alpha=0.25 · RMSE=1.0264 · HitRate@10=0.03237 · cobertura=0.1065 · usuarios evaluados=587


## 5. Ranking, cobertura y usuarios fríos

Tres perfiles: historial amplio (señal colaborativa dominante), historial pequeño (más peso del
contenido) y **usuario nuevo**, que no tiene señal colaborativa y recibe el *fallback* de popularidad,
que **no es personalizado**.

In [7]:
from inf8239_u03.recommenders import ContentRecommender as _CR  # noqa: F401 (uso en el texto)

history = train.groupby("userId").size()
amplio = int(history.idxmax())
pequeno = int(history.idxmin())
nuevo = 999_999

popular_full = weighted_popularity(ratings, movies)
popular_items = popular_full.index.tolist()[:10]
titulo_por_id = movies.set_index("movieId")["title"]

filas = []
for perfil, usuario in [("historial amplio", amplio), ("historial pequeño", pequeno), ("usuario nuevo", nuevo)]:
    if usuario in modelo.user_index:
        top = hybrid_top_n(modelo, train, genre_matrix, movie_row, usuario, 0.75, 10)
        metodo = "híbrido (colaborativo + contenido)"
        for rank, item in enumerate(top.itertuples(), start=1):
            filas.append({"perfil": perfil, "userId": usuario, "rank": rank,
                          "movieId": int(item.movieId), "title": titulo_por_id[int(item.movieId)],
                          "metodo": metodo, "hybrid_score": float(item.hybrid_score)})
    else:
        for rank, movie_id in enumerate(popular_items, start=1):
            filas.append({"perfil": perfil, "userId": usuario, "rank": rank, "movieId": int(movie_id),
                          "title": titulo_por_id[int(movie_id)],
                          "metodo": "fallback de popularidad (NO personalizado)", "hybrid_score": None})

perfiles = pd.DataFrame(filas)
perfiles.to_csv(REPORTS / "profile_recommendations.csv", index=False)
perfiles[["perfil", "rank", "title", "metodo"]].to_string(index=False, max_colwidth=44)

"           perfil  rank                                        title                                     metodo\n historial amplio     1   Life Is Beautiful (La Vita è bella) (1997)         híbrido (colaborativo + contenido)\n historial amplio     2                          12 Angry Men (1957)         híbrido (colaborativo + contenido)\n historial amplio     3                          My Fair Lady (1964)         híbrido (colaborativo + contenido)\n historial amplio     4                          Intouchables (2011)         híbrido (colaborativo + contenido)\n historial amplio     5                            Annie Hall (1977)         híbrido (colaborativo + contenido)\n historial amplio     6               Philadelphia Story, The (1940)         híbrido (colaborativo + contenido)\n historial amplio     7                                Psycho (1960)         híbrido (colaborativo + contenido)\n historial amplio     8 Lives of Others, The (Das leben der Ander...         híbrido (colaborat

## 6. Frontera de Pareto: acierto frente a cobertura y frente a costo

Seis configuraciones sobre el mismo corte temporal: tres tamaños de factorización × dos valores de
`alpha`, más las dos referencias (popularidad y contenido).

In [8]:
import pickle

filas_pareto = []
for factors in (10, 20, 40):
    inicio = time.perf_counter()
    m = MatrixFactorization(factors=factors, seed=RANDOM_STATE).fit(train, epochs=12)
    tiempo = time.perf_counter() - inicio
    tamano = len(pickle.dumps(m)) / 1024
    for alfa in (0.25, 0.75):
        ev = evaluate_hybrid(m, train, test, genre_matrix, movie_row, alfa)
        filas_pareto.append({"configuracion": f"mf{factors}_alpha{alfa}", "familia": "Híbrido",
                             "hit_rate_at_10": ev["hit_rate_at_10"], "catalog_coverage": ev["catalog_coverage"],
                             "rmse": ev["rmse"], "fit_time_s": tiempo, "model_size_kb": tamano})
        print(f"mf{factors}_alpha{alfa}: hit@10={ev['hit_rate_at_10']:.5f} cobertura={ev['catalog_coverage']:.4f} "
              f"tiempo={tiempo:.1f}s tamano={tamano:.0f}KB")

# Referencia 1 · contenido: para cada usuario se recommendations losTitulos similares a su última película
recomendaciones_contenido = {}
for usuario, pelicula in test[["userId", "movieId"]].itertuples(index=False):
    if int(usuario) in recomendaciones_contenido:
        continue
    titulo = titulo_por_id.get(pelicula)
    if titulo is None:
        continue
    try:
        tabla = recomendador.recommend(titulo, 10)
    except KeyError:
        continue
    recomendaciones_contenido[int(usuario)] = [int(i) for i in tabla["movieId"]]

# Referencia 2 · popularidad: la misma lista para todos los usuarios
popular_items = weighted_popularity(ratings, movies).index.tolist()[:10]
recomendaciones_popularidad = {int(u): popular_items for u in test["userId"].unique()}

for nombre, recomendaciones in [("contenido", recomendaciones_contenido),
                                 ("popularidad", recomendaciones_popularidad)]:
    filas_pareto.append({
        "configuracion": nombre,
        "familia": "Referencia",
        "hit_rate_at_10": hit_rate_at_k(recomendaciones, test, 10),
        "catalog_coverage": catalog_coverage(recomendaciones, len(modelo.items)),
        "rmse": float("nan"), "fit_time_s": 0.0, "model_size_kb": 0.0,
    })
    print(f"{nombre}: hit@10={filas_pareto[-1]['hit_rate_at_10']:.5f} "
          f"cobertura={filas_pareto[-1]['catalog_coverage']:.4f}")

pareto = pd.DataFrame(filas_pareto)
pareto.to_csv(REPORTS / "pareto_results.csv", index=False)
pareto

mf10_alpha0.25: hit@10=0.02896 cobertura=0.0838 tiempo=7.4s tamano=1077KB


mf10_alpha0.75: hit@10=0.03066 cobertura=0.0577 tiempo=7.4s tamano=1077KB


mf20_alpha0.25: hit@10=0.03237 cobertura=0.1065 tiempo=7.5s tamano=1883KB


mf20_alpha0.75: hit@10=0.03748 cobertura=0.0775 tiempo=7.5s tamano=1883KB


mf40_alpha0.25: hit@10=0.02726 cobertura=0.1227 tiempo=7.9s tamano=3494KB


mf40_alpha0.75: hit@10=0.03918 cobertura=0.1006 tiempo=7.9s tamano=3494KB


contenido: hit@10=0.00000 cobertura=0.1907
popularidad: hit@10=0.02131 cobertura=0.0010


,configuracion,familia,hit_rate_at_10,catalog_coverage,rmse,fit_time_s,model_size_kb
0,mf10_alpha0.25,Híbrido,0.028961,0.083806,1.033666,7.447715,1077.247070
1,mf10_alpha0.75,Híbrido,0.030664,0.057726,1.033666,7.447715,1077.247070
2,mf20_alpha0.25,Híbrido,0.032368,0.106484,1.026418,7.515890,1882.802734
3,mf20_alpha0.75,Híbrido,0.037479,0.077518,1.026418,7.515890,1882.802734
4,mf40_alpha0.25,Híbrido,0.027257,0.122668,1.021052,7.921060,3493.896484
5,mf40_alpha0.75,Híbrido,0.039182,0.100608,1.021052,7.921060,3493.896484
6,contenido,Referencia,0.000000,0.190702,NaN,0.000000,0.000000
7,popularidad,Referencia,0.021311,0.001031,NaN,0.000000,0.000000


## 7. Figuras de decisión

In [9]:
figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(13.5, 5.2))
eje1.scatter(pareto["fit_time_s"], pareto["hit_rate_at_10"], s=70, color="#b45309")
for _, fila in pareto.iterrows():
    eje1.annotate(fila["configuracion"], (fila["fit_time_s"], fila["hit_rate_at_10"]),
                  textcoords="offset points", xytext=(6, 4), fontsize=8, color="#475569")
eje1.set_xlabel("Tiempo de ajuste (s)")
eje1.set_ylabel("HitRate@10")
eje1.set_title("Utilidad frente a costo de ajuste")
eje1.grid(alpha=0.25)

eje2.scatter(pareto["catalog_coverage"], pareto["hit_rate_at_10"], s=70, color="#0f7c7b")
for _, fila in pareto.iterrows():
    eje2.annotate(fila["configuracion"], (fila["catalog_coverage"], fila["hit_rate_at_10"]),
                  textcoords="offset points", xytext=(6, 4), fontsize=8, color="#475569")
eje2.set_xlabel("Cobertura del catálogo")
eje2.set_ylabel("HitRate@10")
eje2.set_title("Utilidad frente a cobertura del catálogo")
eje2.grid(alpha=0.25)
figura.suptitle("MovieLens · acierto, cobertura y costo de las configuraciones")
figura.tight_layout()
figura.savefig(REPORTS / "pareto_ejercicio05.png", dpi=170, bbox_inches="tight")
plt.close(figura)
print("Guardado:", REPORTS / "pareto_ejercicio05.png")

Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_3/INF8239_U03_Recomendadores/reports/pareto_ejercicio05.png


In [10]:
resumen = {
    "dataset": "MovieLens Latest Small (GroupLens Research, University of Minnesota)",
    "licencia": "uso académico con cita (Harper y Konstan, 2015, DOI 10.1145/2827872)",
    "redistribucion": "no se redistribuye: se descarga por script y no se versiona",
    "protocolo": "corte temporal leave-one-out, semilla 42",
    "configuraciones": pareto.dropna(subset=["hit_rate_at_10"]).to_dict(orient="records"),
}
(REPORTS / "ejercicio05_resumen.json").write_text(json.dumps(resumen, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in resumen.items() if k != "configuraciones"}, indent=2))

{
  "dataset": "MovieLens Latest Small (GroupLens Research, University of Minnesota)",
  "licencia": "uso acad\u00e9mico con cita (Harper y Konstan, 2015, DOI 10.1145/2827872)",
  "redistribucion": "no se redistribuye: se descarga por script y no se versiona",
  "protocolo": "corte temporal leave-one-out, semilla 42"
}


## 8. Interpretación y decisión final

**Observación.** El baseline de popularidad concentra las primeras 10 posiciones enfilms por defecto
(películas con muchas valoraciones) y no aporta personalización: es el mínimo que un sistema útil debe
superar. El recomendador por contenido devuelve películas con **el mismo conjunto de géneros** que la
consulta, con `content_score` 1,0, es decir, mide coincidencia de categorías, no gusto.

**Evidencia.** El híbrido con `alpha=0,75` alcanza el mayor HitRate@10 y la menor cobertura del
catálogo; con `alpha=0,25` baja el acierto y sube la cobertura. Los dos effects son el mismo
phenómeno: concentrates las recomendaciones cuando pesa más la señal colaborativa. El usuario frío
(999999) no tiene señal colaborativa y recibe la lista de popularidad, **idéntica para todos y no
personalizada**, que es el comportamiento honesto del sistema, no un error.

**Decisión.** Se entrega el híbrido con `alpha=0,75` y 20 factores: es la configuración con mejor
acierto en el Top-10 y su costo (tiempo y tamaño) está en el mismo orden de magnitud que el resto de
la frontera. Si el objetivo de negocio fuera cobertura y no acierto, la decisión cambiaría a
`alpha=0,25`. El límite es que **ninguna de estas señales es una preferencia**: la cobertura se
reporta siempre junto al HitRate@10 para hacer visible la burbuja de filtro.